# Daily Customer RFM Pipeline

This notebook runs the complete daily pipeline:

1. Load database credentials from environment variables.
2. Read non-cancelled orders from PostgreSQL.
3. Calculate recency, frequency, and monetary metrics.
4. Create 1–5 RFM scores and customer segments.
5. Prepare an idempotent upsert into `ecom.customer_rfm_daily`.


In [1]:
# Standard libraries
from datetime import date
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Set the run date for the RFM analysis
run_date = date.today()

# Custom modules
from src.db import get_connection, test_connection
from src.rfm import calculate_rfm_scores
from src.upsert import upsert_rfm_results

# Data analysis
import pandas as pd

In [2]:
# Connect using read credentials

read_conn = get_connection("read")

print(
    "Connection successful:",
    test_connection(read_conn)
)

Connection successful: True


In [3]:
# Load customer RFM metrics query

sql_file = (
    project_root
    / "sql"
    / "customer_rfm_metrics.sql"
)

with open(sql_file, "r") as file:
    query = file.read()

print("Query loaded successfully.")

Query loaded successfully.


In [4]:
# Execute query and load data into dataframe

cursor = read_conn.cursor()

cursor.execute(
    query,
    {"run_date": run_date}
)

rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]

rfm_df = pd.DataFrame(
    rows,
    columns=columns
)

print(rfm_df.shape)

rfm_df.head()

(8438, 5)


,customer_id,last_order_date,recency_days,frequency_orders,monetary_value
0,1,2026-05-25 07:57:11+00:00,27,7,88185.68
1,2,2026-05-07 05:03:47+00:00,45,1,3181.28
2,3,2026-05-25 08:00:08+00:00,27,1,9209.46
3,4,2026-05-27 10:19:04+00:00,25,4,28310.56
4,5,2026-04-12 16:53:45+00:00,70,1,10553.92


In [5]:
# Create RFM scores and customer segments

rfm_df = calculate_rfm_scores(
    rfm_df,
    run_date
)

rfm_df.head()

,customer_id,last_order_date,recency_days,frequency_orders,monetary_value,run_date,r_score,f_score,m_score,rfm_score,rfm_segment
0,1,2026-05-25 07:57:11+00:00,27,7,88185.68,2026-06-21,4,5,5,455,Champions
1,2,2026-05-07 05:03:47+00:00,45,1,3181.28,2026-06-21,3,1,2,312,Others
2,3,2026-05-25 08:00:08+00:00,27,1,9209.46,2026-06-21,4,1,3,413,Others
3,4,2026-05-27 10:19:04+00:00,25,4,28310.56,2026-06-21,4,4,4,444,Champions
4,5,2026-04-12 16:53:45+00:00,70,1,10553.92,2026-06-21,2,1,3,213,Hibernating


In [6]:
print(rfm_df.shape)

rfm_df.columns.tolist()

(8438, 11)


['customer_id',
 'last_order_date',
 'recency_days',
 'frequency_orders',
 'monetary_value',
 'run_date',
 'r_score',
 'f_score',
 'm_score',
 'rfm_score',
 'rfm_segment']

In [7]:
rfm_df["rfm_segment"].value_counts()

rfm_segment
Others          2036
Champions       1846
Hibernating     1748
At Risk         1338
Loyal            941
Big Spenders     529
Name: count, dtype: int64

In [8]:
# Enable only when write credentials are available

WRITE_ENABLED = False

if WRITE_ENABLED:
    # Create write connection

    write_conn = get_connection("write")

    upsert_rfm_results(
        write_conn,
        rfm_df
    )

else:

    print(
        f"Dry run: {len(rfm_df)} rows prepared. Write skipped."
    )

Dry run: 8438 rows prepared. Write skipped.


In [9]:
# Close Connections

read_conn.close()

if WRITE_ENABLED:
    write_conn.close()

print("Pipeline completed successfully.")

Pipeline completed successfully.
